# 05 Final Load Prep

**Objective:** Calculate final KPIs and prepare Tableau-ready export files for the Banking Customer Churn Analysis project.

This notebook uses the cleaned dataset from `../data/churn_clean.csv`, standardizes column names, calculates business KPIs, and saves summary tables for Tableau dashboards.


## 1. Load Data

Load the cleaned churn dataset, standardize column names, and preview the data before KPI calculations.


In [1]:
# Import required libraries for data processing and KPI calculations.
import pandas as pd
import numpy as np
import os

# Load the cleaned dataset from the project data folder.
df = pd.read_csv('../data/churn_clean.csv')

# Standardize column names for consistent calculations across the notebook.
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Display the dataset shape and first five rows for validation.
print('Dataset Shape:', df.shape)
df.head()






Dataset Shape: (28382, 26)


,vintage,age,gender,dependents,occupation,customer_nw_category,current_balance,previous_month_end_balance,average_monthly_balance_prevq,average_monthly_balance_prevq2,...,churn,last_transaction,transaction_year,transaction_month,transaction_day,no_transaction_flag,age_group,balance_segment,total_transactions,activity_level
0,2101,66,Male,0.0,self_employed,2,1458.71,1458.71,1458.71,1449.07,...,0,2019-05-21,2019.0,5.0,21.0,0,Senior,Medium,0.40,Low Activity
1,2348,35,Male,0.0,self_employed,2,5390.37,8704.66,7799.26,12419.41,...,0,2019-11-01,2019.0,11.0,1.0,0,Adult,Medium,5486.83,Moderate
2,2194,31,Male,0.0,salaried,2,3913.16,5815.29,4910.17,2815.94,...,0,NaN,NaN,NaN,NaN,1,Adult,Medium,6047.34,Moderate
3,2329,90,Male,0.0,self_employed,2,2291.91,2291.91,2084.54,1006.54,...,1,2019-08-06,2019.0,8.0,6.0,0,Senior,Medium,0.94,Low Activity
4,1579,42,Male,2.0,self_employed,3,927.72,1401.72,1643.31,1871.12,...,1,2019-11-03,2019.0,11.0,3.0,0,Middle Age,Low,588.95,Low Activity


## 2. KPI Calculations

Calculate core churn KPIs and segment-level churn rates needed for business analysis and Tableau reporting.


In [2]:
# Calculate overall customer counts and churn metrics.
total_customers = len(df)
total_churned = int(df['churn'].sum())
total_retained = int(total_customers - total_churned)
overall_churn_rate = df['churn'].mean() * 100

# Calculate average metrics for churned and retained customers.
avg_current_balance = df.groupby('churn', observed=False)['current_balance'].mean().rename({0: 'Retained', 1: 'Churned'})
avg_vintage = df.groupby('churn', observed=False)['vintage'].mean().rename({0: 'Retained', 1: 'Churned'})
avg_total_transactions = df.groupby('churn', observed=False)['total_transactions'].mean().rename({0: 'Retained', 1: 'Churned'})

# Calculate churn rates by important customer segments.
churn_by_age_group = (df.groupby('age_group', observed=False)['churn'].mean() * 100).round(2)
churn_by_gender = (df.groupby('gender', observed=False)['churn'].mean() * 100).round(2)
churn_by_occupation = (df.groupby('occupation', observed=False)['churn'].mean() * 100).round(2)
churn_by_balance_segment = (df.groupby('balance_segment', observed=False)['churn'].mean() * 100).round(2)
churn_by_activity_level = (df.groupby('activity_level', observed=False)['churn'].mean() * 100).round(2)
churn_by_customer_nw_category = (df.groupby('customer_nw_category', observed=False)['churn'].mean() * 100).round(2)

# Print headline KPIs for quick review.
print(f'Total Customers: {total_customers}')
print(f'Total Churned: {total_churned}')
print(f'Total Retained: {total_retained}')
print(f'Overall Churn Rate: {overall_churn_rate:.2f}%')

print('\nAvg Current Balance — Churned vs Retained:')
print(avg_current_balance.round(2))

print('\nAvg Vintage — Churned vs Retained:')
print(avg_vintage.round(2))

print('\nAvg Total Transactions — Churned vs Retained:')
print(avg_total_transactions.round(2))

print('\nChurn Rate by Age Group (%):')
print(churn_by_age_group)

print('\nChurn Rate by Gender (%):')
print(churn_by_gender)

print('\nChurn Rate by Occupation (%):')
print(churn_by_occupation)

print('\nChurn Rate by Balance Segment (%):')
print(churn_by_balance_segment)

print('\nChurn Rate by Activity Level (%):')
print(churn_by_activity_level)

print('\nChurn Rate by Customer NW Category (%):')
print(churn_by_customer_nw_category)





Total Customers: 28382
Total Churned: 5260
Total Retained: 23122
Overall Churn Rate: 18.53%

Avg Current Balance — Churned vs Retained:
churn
Retained    6734.15
Churned     4202.91
Name: current_balance, dtype: float64

Avg Vintage — Churned vs Retained:
churn
Retained    2091.76
Churned     2088.42
Name: vintage, dtype: float64

Avg Total Transactions — Churned vs Retained:
churn
Retained    3515.49
Churned     7908.48
Name: total_transactions, dtype: float64

Churn Rate by Age Group (%):
age_group
Adult         19.98
Middle Age    18.79
Senior        16.72
Young         17.51
Name: churn, dtype: float64

Churn Rate by Gender (%):
gender
Female    17.55
Male      19.18
Name: churn, dtype: float64

Churn Rate by Occupation (%):
occupation
Unknown          16.25
company          10.00
retired          15.07
salaried         17.11
self_employed    19.84
student          15.74
Name: churn, dtype: float64

Churn Rate by Balance Segment (%):
balance_segment
High      12.18
Low       65.31


## 3. Save Tableau Export Tables

Create clean summary tables and save them as CSV files for Tableau dashboard building.


In [3]:
# Create the Tableau export folder using pandas' bundled OS utility to avoid extra imports.
export_dir = '../data/tableau_exports'
os.makedirs(export_dir, exist_ok=True)

# Create a KPI summary table with key project metrics.
kpi_summary = pd.DataFrame({
    'metric': [
        'Total Customers',
        'Total Churned',
        'Total Retained',
        'Overall Churn Rate %',
        'Avg Current Balance - Churned',
        'Avg Current Balance - Retained',
        'Avg Vintage - Churned',
        'Avg Vintage - Retained',
        'Avg Total Transactions - Churned',
        'Avg Total Transactions - Retained'
    ],
    'value': [
        total_customers,
        total_churned,
        total_retained,
        round(overall_churn_rate, 2),
        round(avg_current_balance['Churned'], 2),
        round(avg_current_balance['Retained'], 2),
        round(avg_vintage['Churned'], 2),
        round(avg_vintage['Retained'], 2),
        round(avg_total_transactions['Churned'], 2),
        round(avg_total_transactions['Retained'], 2)
    ]
})

# Define a helper function to build segment-level Tableau tables.
def build_segment_table(segment_col):
    # Aggregate customer count, churned count, retained count, and churn rate by segment.
    table = df.groupby(segment_col, observed=False).agg(
        total_customers=('churn', 'count'),
        churned_customers=('churn', 'sum')
    ).reset_index()
    table['retained_customers'] = table['total_customers'] - table['churned_customers']
    table['churn_rate_%'] = (table['churned_customers'] / table['total_customers'] * 100).round(2)
    return table

# Build required Tableau export tables.
age_group_table = build_segment_table('age_group')
gender_table = build_segment_table('gender')
occupation_table = build_segment_table('occupation')
balance_segment_table = build_segment_table('balance_segment')
activity_level_table = build_segment_table('activity_level')

# Create vintage buckets in months equivalent from vintage days for Tableau segmentation.
df['vintage_months'] = (df['vintage'] / 30).round(1)
df['vintage_bucket'] = pd.cut(
    df['vintage_months'],
    bins=[0, 12, 36, 60, np.inf],
    labels=['New(0-12)', 'Growing(12-36)', 'Loyal(36-60)', 'Champion(60+)'],
    right=False
)
vintage_bucket_table = build_segment_table('vintage_bucket')

# Save all KPI and segment tables for Tableau.
kpi_summary.to_csv(f'{export_dir}/kpi_summary.csv', index=False)
age_group_table.to_csv(f'{export_dir}/churn_by_age_group.csv', index=False)
gender_table.to_csv(f'{export_dir}/churn_by_gender.csv', index=False)
occupation_table.to_csv(f'{export_dir}/churn_by_occupation.csv', index=False)
balance_segment_table.to_csv(f'{export_dir}/churn_by_balance_segment.csv', index=False)
activity_level_table.to_csv(f'{export_dir}/churn_by_activity_level.csv', index=False)
vintage_bucket_table.to_csv(f'{export_dir}/churn_by_vintage_bucket.csv', index=False)

# Confirm that all summary files have been created.
print('All KPI tables saved to:', export_dir)





All KPI tables saved to: ../data/tableau_exports


## 4. Save Master Tableau File

Save the complete cleaned dataset with final derived fields for Tableau dashboard development.


In [4]:
# Save the full prepared dataframe as the master Tableau file.
df.to_csv(f'{export_dir}/churn_final_tableau.csv', index=False)

# Confirm master file export.
print('Tableau master file saved:', f'{export_dir}/churn_final_tableau.csv')





Tableau master file saved: ../data/tableau_exports/churn_final_tableau.csv


## 5. Final Summary

Print the final project-readiness checklist for Tableau dashboard work.


In [6]:
# Print final notebook summary for quick project validation.
print(f'Total Customers: {total_customers}')
print(f'Churn Rate: {overall_churn_rate:.2f}%')
print('All KPI tables saved')
print('Tableau master file saved')
print('Ready for Tableau!')





Total Customers: 28382
Churn Rate: 18.53%
All KPI tables saved
Tableau master file saved
Ready for Tableau!
